In [25]:
import argparse
import json
import os
import tempfile
import numpy as np
import subprocess
from tqdm import tqdm
import pandas as pd
import datetime
import time
import gc
import random
import sys
import shutil
import glob

sys.path.append("/home/agustin/phd/synthesis")
sys.path.append(
    "/home/agustin/phd/miccai/miccai_2026/mri_x_fields/experiments/test4_finetune_brainst_diffusion_model/training/networks_declaration"
)

import utils.functions as fc
import utils.nifti_functions as nfc


# pytorch
import torch
from torch.amp import GradScaler, autocast
from torch.utils.tensorboard import SummaryWriter
from torch.utils.checkpoint import checkpoint
from torch.utils.data import Dataset, Sampler
import torch.distributed as dist



In [26]:
class LoadPaths:
    def __init__(
        self,
        dataset_path_name,
        used_modalities,
        used_resolutions,
        nb_seg_classes,
        dataset_filters=None,
        max_subjects=None,
    ):
        """Load the dataset and latents from the specified paths.
        Args:
          - dataset_path_name: Path to the dataset.
          - conditions_keys_ordered: List of condition keys in the desired order.
          - dataset_filters: Optional filters to apply to the dataset in the form of a dictionary where keys are column names and values are lists of values to filter by.
          - att_mask_path: Path to the attention masks.
          - att_mask_resolution_list: List of resolutions for the attention masks.
          - att_mask_structure_mapping: Mapping of the attention mask structure.
        """
        # self.complete_dataset = load_dataset.LoadDataset(training_dataset_path_name)
        self.df = pd.read_csv(dataset_path_name)

        if dataset_filters is not None:
            for column, values_list in dataset_filters.items():
                self.df = self.df[self.df[column].isin(values_list)]

        self.df = self.df[
            (self.df["modality"].isin(used_modalities))
            & (self.df["resolution"].isin(used_resolutions))
        ]

        self.modality_index_mapping = {
            modality: idx for idx, modality in enumerate(used_modalities)
        }
        self.resolution_index_mapping = {
            resolution: idx for idx, resolution in enumerate(used_resolutions)
        }

        # map modality to index
        self.df["modality_idx"] = self.df["modality"].map(self.modality_index_mapping)
        self.df["resolution_idx"] = self.df["resolution"].map(
            self.resolution_index_mapping
        )

        if max_subjects is not None:
            # self.df = self.df[self.df["subject_id"].isin(self.df["subject_id"].unique()[:max_subjects])]
            self.df = self.df.head(max_subjects)

        self.nb_seg_classes = nb_seg_classes

    def get_data(self, split="train"):
        complete_df = self.df.copy()
        if split is not None:
            complete_df = complete_df[complete_df["split"] == split]

        # order by [sid, resolution, modality, ]
        complete_df = complete_df.sort_values(by=["sid", "modality", "resolution"])

        # self.subject_ids = complete_df["sid"].unique()

        instances = []
        for i, row in complete_df.iterrows():
            instance_dict = {}
            latent_path = row["latent_path"]
            seg_row = f"latent_seg_supersynth_merged_{self.nb_seg_classes}_path" if f"latent_seg_supersynth_merged_{self.nb_seg_classes}_path" in row else f"latent_seg_synthseg_merged_{self.nb_seg_classes}_path"
            # verify that the latent path exists
            if not os.path.exists(latent_path):
                # print(f"Latent path {latent_path} does not exist. Skipping this instance.")
                continue
            instance_dict["latent_path"] = latent_path
            instance_dict["latent_synthsr_path"] = row["latent_synthsr_path"]
            instance_dict["subject_id"] = row["sid"]
            instance_dict["modality"] = row["modality"]
            instance_dict["modality_idx"] = row["modality_idx"]
            instance_dict["resolution"] = row["resolution"]
            instance_dict["resolution_idx"] = row["resolution_idx"]
            instance_dict["segmentation_npy_path"] = row[seg_row]
            instance_dict["org_img_path"] = row["org_img_path"]
            instance_dict["seg_supersynth_path"] = row["seg_supersynth_path"]
            instances.append(instance_dict)
        return instances




In [27]:
# class PrepareDataset(Dataset):
#     def __init__(
#         self,
#         dataset_path_name,
#         used_modalities,
#         used_resolutions,
#         dataset_filters=None,
#         split="train",
#         max_subjects=None,
#         nb_seg_classes=3,
#         patch_size=(96, 96, 96),
#         used_pondered_segmentation=False,
#         pondered_seg_generator=None,
#         latent_synthsr_generator=None,
#         latent_synthsr_prob=None,
#     ):

#         data_loader = LoadPaths(
#             dataset_path_name,
#             used_modalities=used_modalities,
#             used_resolutions=used_resolutions,
#             dataset_filters=dataset_filters,
#             max_subjects=max_subjects,
#             nb_seg_classes=nb_seg_classes,
#         )

#         self.train_data = data_loader.get_data(split=split)

#         print(f"Number of {split} images: {len(self.train_data)}")

#         self.patch_size = patch_size
#         self.nb_seg_classes = nb_seg_classes
#         self.used_modalities = used_modalities

#         self.use_pondered_segmentation = used_pondered_segmentation
#         self.pondered_seg_generator = pondered_seg_generator

#         self.latent_synthsr_generator = latent_synthsr_generator
#         self.latent_synthsr_prob = latent_synthsr_prob

#         # ✅ CACHE per worker (important)
#         self.image_cache = {}
#         self.seg_cache = {}
        
#                 # number of latent in the folder
#         self.num_instances = len(self.train_data)
#         self._length = self.num_instances
        
    
#     def __len__(self):
#         return self._length


#     # def load_image(self, path):
#     #     img = np.load(path).astype(np.float32)  # (D,H,W)
#     #     img = torch.from_numpy(img).unsqueeze(0)  # (1,D,H,W)
#     #     return img
    
#     def load_segmentation(self, segmentation_npy_path):
#         if segmentation_npy_path.endswith(".npy"):
#             segmentation = np.load(segmentation_npy_path)
#         elif segmentation_npy_path.endswith(".nii") or segmentation_npy_path.endswith(".nii.gz"):
#             segmentation = nfc.load_nifti(segmentation_npy_path)[0]
#         # convert into one-hot encoding
#         unique_labels = list(
#             range(1, self.nb_seg_classes + 1)
#         )  # assuming classes are labeled from 1 to nb_classes

#         seg_onehot = []
#         for label in unique_labels:
#             seg_onehot.append(np.where(segmentation == label, 1.0, 0.0))
#         seg_onehot = np.stack(seg_onehot, axis=0)  # (C, D, H, W)

#         return torch.from_numpy(seg_onehot).float()
        
#     def load_image(self, path):
#         if path.endswith(".npy"):
#             img = np.load(path).astype(np.float32)  # (D,H,W)
#         elif path.endswith(".nii") or path.endswith(".nii.gz"):
#             img = nfc.load_nifti(path)[0]
#         img = torch.from_numpy(img).unsqueeze(0)  # (1,D,H,W)
#         return img
    
#     # def load_segmentation_npy(self, path):
#     #     seg = np.load(path).astype(np.float32)  # (D,H,W)
        
    
#     # def random_crop(self, volume, segmentation=None):
#     #     _, D, H, W = volume.shape
#     #     pd, ph, pw = self.patch_size

#     #     z = random.randint(0, D - pd)
#     #     y = random.randint(0, H - ph)
#     #     x = random.randint(0, W - pw)

#     #     if segmentation is not None:
#     #         return volume[:, z:z+pd, y:y+ph, x:x+pw], segmentation[:, z:z+pd, y:y+ph, x:x+pw]
#     #     return volume[:, z:z+pd, y:y+ph, x:x+pw]
    
#     def random_crop(self, volume, segmentation=None):
#         _, D, H, W = volume.shape
#         pd, ph, pw = self.patch_size

#         # -----------------------
#         # ensure valid bounds
#         # -----------------------
#         max_z = D - pd
#         max_y = H - ph
#         max_x = W - pw

#         z = random.randint(0, max_z)
#         y = random.randint(0, max_y)
#         x = random.randint(0, max_x)

#         patch_img = volume[:, z:z+pd, y:y+ph, x:x+pw]

#         if segmentation is None:
#             return patch_img

#         patch_seg = segmentation[:, z:z+pd, y:y+ph, x:x+pw]

#         return patch_img, patch_seg

#     def __getitem__(self, index):

#         instance = self.train_data[index]

#         subject_id = instance["subject_id"]
#         img_path = instance["org_img_path"]
#         # seg_path = instance["segmentation_npy_path"]
#         seg_path = instance["seg_supersynth_path"]

#         # -----------------------
#         # IMAGE (with cache)
#         # -----------------------
#         if img_path not in self.image_cache:
#             self.image_cache[img_path] = self.load_image(img_path)

#         full_img = self.image_cache[img_path]

#         # -----------------------
#         # SEGMENTATION (with cache)
#         # -----------------------
#         if seg_path not in self.seg_cache:
#             self.seg_cache[seg_path] = self.load_segmentation(seg_path)

#         full_seg = self.seg_cache[seg_path]

#         # -----------------------
#         # SAMPLE PATCH
#         # -----------------------
#         img_patch, seg_patch = self.random_crop(full_img, full_seg)
        
#         # print(f"img_patch shape: {img_patch.shape}, seg_patch shape: {seg_patch.shape}")


#         example = {
#             "subject_id": subject_id,
#             "image": img_patch,
#             "segmentation": seg_patch,
#             "modality": instance["modality"],
#             "modality_idx": torch.tensor(instance["modality_idx"]),
#             "resolution_idx": torch.tensor(instance["resolution_idx"]),
#         }
        
#         # number of images in cache.
#         print(f"Number of images in cache: {len(self.image_cache)}")

#         return example

In [ ]:
class PrepareDataset(Dataset):
    def __init__(
        self,
        dataset_path_name,
        used_modalities,
        used_resolutions,
        dataset_filters=None,
        split="train",
        max_subjects=None,
        nb_seg_classes=3,
        patch_size=(96, 96, 96),
        used_pondered_segmentation=False,
        pondered_seg_generator=None,
        latent_synthsr_generator=None,
        latent_synthsr_prob=None,
        ensure_info_percent=None
    ):

        data_loader = LoadPaths(
            dataset_path_name,
            used_modalities=used_modalities,
            used_resolutions=used_resolutions,
            dataset_filters=dataset_filters,
            max_subjects=max_subjects,
            nb_seg_classes=nb_seg_classes,
        )

        self.train_data = data_loader.get_data(split=split)

        print(f"Number of {split} images: {len(self.train_data)}")

        self.patch_size = patch_size
        self.nb_seg_classes = nb_seg_classes
        self.used_modalities = used_modalities

        self.use_pondered_segmentation = used_pondered_segmentation
        self.pondered_seg_generator = pondered_seg_generator

        self.latent_synthsr_generator = latent_synthsr_generator
        self.latent_synthsr_prob = latent_synthsr_prob
        
        # number of latent in the folder
        self.num_instances = len(self.train_data)
        self._length = self.num_instances
        
        self.ensure_info_percent = ensure_info_percent
    def __len__(self):
        return self._length


    # def load_image(self, path):
    #     img = np.load(path).astype(np.float32)  # (D,H,W)
    #     img = torch.from_numpy(img).unsqueeze(0)  # (1,D,H,W)
    #     return img
    
    def load_segmentation(self, segmentation_npy_path):
        if segmentation_npy_path.endswith(".npy"):
            segmentation = np.load(segmentation_npy_path)
        elif segmentation_npy_path.endswith(".nii") or segmentation_npy_path.endswith(".nii.gz"):
            segmentation = nfc.load_nifti(segmentation_npy_path)[0]
        # convert into one-hot encoding
        unique_labels = list(
            range(1, self.nb_seg_classes + 1)
        )  # assuming classes are labeled from 1 to nb_classes

        seg_onehot = []
        for label in unique_labels:
            seg_onehot.append(np.where(segmentation == label, 1.0, 0.0))
        seg_onehot = np.stack(seg_onehot, axis=0)  # (C, D, H, W)

        return torch.from_numpy(seg_onehot).float()
        
    def load_image(self, path):
        if path.endswith(".npy"):
            img = np.load(path).astype(np.float32)  # (D,H,W)
        elif path.endswith(".nii") or path.endswith(".nii.gz"):
            img = nfc.load_nifti(path)[0]
        img = torch.from_numpy(img).unsqueeze(0)  # (1,D,H,W)
        return img
    

    def __random_crop(self, volume, segmentation=None):
        _, D, H, W = volume.shape
        pd, ph, pw = self.patch_size

        # -----------------------
        # ensure valid bounds
        # -----------------------
        max_z = D - pd
        max_y = H - ph
        max_x = W - pw

        z = random.randint(0, max_z)
        y = random.randint(0, max_y)
        x = random.randint(0, max_x)

        patch_img = volume[:, z:z+pd, y:y+ph, x:x+pw]
        patch_seg = None

        if segmentation is not None:
            patch_seg = segmentation[:, z:z+pd, y:y+ph, x:x+pw]

        return patch_img, patch_seg
    
    def random_crop(self, volume, segmentation=None):
        counter = 0
        return_pathch = False
        while True:
            patch_img, patch_seg = self.__random_crop(volume, segmentation)

            if self.ensure_info_percent is not None and segmentation is not None:
                mask = patch_img > 0  # Example condition, adjust as needed
                # esure that the background is less than ensure_info_percent
                if mask.sum() / mask.numel() >= self.ensure_info_percent:
                    return_pathch = True
            else:
                return_pathch = True
                
            counter += 1
            if counter > 100:  # Prevent infinite loop
                print("FAILED to find a valid patch after 100 attempts. Returning the last sampled patch.")
                return_pathch = True
                
            if return_pathch:
                return patch_img, patch_seg
            
    def __getitem__(self, index):

        instance = self.train_data[index]
        example = {}
        example["subject_id"] = instance["subject_id"]

        example["modality"] = instance["modality"]
        example["modality_idx"] = torch.tensor(instance["modality_idx"])
        example["resolution"] = instance["resolution"]
        example["resolution_idx"] = torch.tensor(instance["resolution_idx"])
        example["org_img_path"] = instance["org_img_path"]


        img_path = instance["org_img_path"]
        seg_path = instance["seg_supersynth_path"]

        # -----------------------
        # IMAGE (with cache)
        # -----------------------
        full_img = self.load_image(img_path)
        full_seg = self.load_segmentation(seg_path)

        # -----------------------
        # SAMPLE PATCH
        # -----------------------
        img_patch, seg_patch = self.random_crop(full_img, full_seg)
        
        example["img_patch"] = img_patch
        example["seg_patch"] = seg_patch
        
        return example

In [32]:
    
def collate_fn(examples):
    res_dict = {}

    res_dict["subject_id"] = [ex["subject_id"] for ex in examples]
    res_dict["modality"] = [ex["modality"] for ex in examples]

    modality_idx = torch.stack([ex["modality_idx"] for ex in examples])
    res_dict["modality_idx"] = modality_idx.contiguous().long()

    resolution_idx = torch.stack([ex["resolution_idx"] for ex in examples])
    res_dict["resolution_idx"] = resolution_idx.contiguous().long()

    # -------------------------
    # IMAGE PATCH (NEW)
    # -------------------------
    image = torch.stack([ex["image"] for ex in examples])
    res_dict["image"] = image.contiguous().float()

    # -------------------------
    # SEGMENTATION PATCH
    # -------------------------
    segmentation = torch.stack([ex["segmentation"] for ex in examples])
    res_dict["segmentation"] = segmentation.contiguous().float()

    return res_dict




def instantiate_dataset(
    dataset_path_name,
    used_modalities,
    used_resolutions,
    batch_size,
    dataset_filters=None,
    split="train",
    max_subjects=None,
    nb_seg_classes=3,
    used_pondered_segmentation=False,
    pondered_seg_generator=None,
    latent_synthsr_generator=None,
    latent_synthsr_prob=None,
    # models=None,
    # dm_nb_inference_steps=4
    ensure_info_percent=None
):
    # ---- Data set creation
    train_dataset = PrepareDataset(
        dataset_path_name=dataset_path_name,
        used_modalities=used_modalities,
        used_resolutions=used_resolutions,
        dataset_filters=dataset_filters,
        split=split,
        max_subjects=max_subjects,
        nb_seg_classes=nb_seg_classes,
        used_pondered_segmentation=used_pondered_segmentation,
        pondered_seg_generator=pondered_seg_generator,
        latent_synthsr_generator=latent_synthsr_generator,
        latent_synthsr_prob=latent_synthsr_prob,
        # models=models,
        # dm_nb_inference_steps=dm_nb_inference_steps
        ensure_info_percent=ensure_info_percent
    )

    # sampler = MaxPerSubjectSampler(train_dataset, max_per_subject=max_timepoints_per_epoch, shuffle=True, generator=gen_dataloader)

    train_dataloader = torch.utils.data.DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=split == "train",  # shuffle only in training
        # sampler=sampler,
        collate_fn=lambda examples: collate_fn(examples),
        # generator=gen_dataloader,
        num_workers=2,
        persistent_workers=True,
    )
    return train_dataloader



In [38]:
args_train = {
    # directories
    "df_path": "/home/agustin/phd/miccai/miccai_2026/mri_x_fields/data/csv/train_data.csv",

    "batch_size": 2,  # 3

    
 # "validation_first": True, # if True, the model will be validated before the first training step, if False, the model will be validated after the first training step
    "val_seeds": [42],  # [0,12357], # seeds for the noise generation during validation
    "max_val_subjects": None,  # max number of subjects to be generated during validation, set to None to use all the subjects in the val dataloader
    "val_dataset_filters": {
        "sid": ["S0006"]  # Example filter for a specific subject ID
    },

    "used_modalities": ["T1W", "T2W", "T2FLAIR"],  # "T1W", "T2W", "T2FLAIR"
    "used_resolutions": [0.1, 1.5, 3, 5, 7],  # 0.1, 1.5, 3, 5, 7

    "nb_seg_classes": 3,  # number of segmentation classes including the background, used for the one hot encoding of the segmentation maps
    "used_pondered_segmentation": True,  # if True, the segmentation maps are pondered by the distance to the borders of the structures, if False, the segmentation maps are one hot encoded without pondering

    "latent_synthsr_prob": None,


}


args_train = fc.dict_to_args(args_train, deep_conversion=True)


train_dataloader = instantiate_dataset(
    dataset_path_name=args_train.df_path,
    used_modalities=args_train.used_modalities,
    used_resolutions=args_train.used_resolutions,
    # latents_path=args_train.latents_path,
    batch_size=args_train.batch_size,
    split="train",
    nb_seg_classes=args_train.nb_seg_classes,
    used_pondered_segmentation=args_train.used_pondered_segmentation,
    pondered_seg_generator=None,
    latent_synthsr_generator=None,
    latent_synthsr_prob=args_train.latent_synthsr_prob,
    # models = models_dict,
    # dm_nb_inference_steps=args_train.dm_nb_inference_steps
    max_subjects=10,
    ensure_info_percent=0.10

)


for epoch in range(0, 1):
    for batch in train_dataloader:

        # prepare inputs
        images = batch["image"]
        segmentation = batch["segmentation"]

        condition_modality_idx = batch["modality_idx"]
        condition_resolution_idx = batch["resolution_idx"]
        
        print(f"Images shape: {images.shape}, Segmentation shape: {segmentation.shape}")
        # break  # Just to demonstrate one batch

Number of train images: 10
Images shape: torch.Size([2, 1, 96, 96, 96]), Segmentation shape: torch.Size([2, 3, 96, 96, 96])
Images shape: torch.Size([2, 1, 96, 96, 96]), Segmentation shape: torch.Size([2, 3, 96, 96, 96])
Images shape: torch.Size([2, 1, 96, 96, 96]), Segmentation shape: torch.Size([2, 3, 96, 96, 96])
Images shape: torch.Size([2, 1, 96, 96, 96]), Segmentation shape: torch.Size([2, 3, 96, 96, 96])
Images shape: torch.Size([2, 1, 96, 96, 96]), Segmentation shape: torch.Size([2, 3, 96, 96, 96])


In [39]:
output_path = "/home/agustin/phd/miccai/miccai_2026/mri_x_fields/experiments/test8_patch_refinement/training/scripts/create_dataloader/output"
np.save(os.path.join(output_path, "images.npy"), images.numpy()[0,0])

In [ ]:

# class PrepareDataset(Dataset):
#     def __init__(
#         self,
#         dataset_path_name,
#         used_modalities,
#         used_resolutions,
#         dataset_filters=None,
#         split="train",
#         max_subjects=None,
#         nb_seg_classes=3,
#         used_pondered_segmentation=False,
#         pondered_seg_generator=None,
#         latent_synthsr_generator=None,
#         latent_synthsr_prob=None,
#         # models=None,
#         # dm_nb_inference_steps=4,
#     ):

#         # load data
#         data_loader = LoadPaths(
#             dataset_path_name,
#             used_modalities=used_modalities,
#             used_resolutions=used_resolutions,
#             dataset_filters=dataset_filters,
#             max_subjects=max_subjects,
#             nb_seg_classes=nb_seg_classes,
#         )

#         self.train_data = data_loader.get_data(split=split)


#         print(f"Number of {split} images: {len(self.train_data)}")

#         # number of latent in the folder
#         self.num_instances = len(self.train_data)
#         self._length = self.num_instances
#         self.nb_seg_classes = nb_seg_classes
#         self.used_modalities = used_modalities
#         self.use_pondered_segmentation = used_pondered_segmentation
#         self.pondered_seg_generator = pondered_seg_generator
#         self.split = split
#         # self.ids_list = list(self.train_data.keys())

#         self.latent_synthsr_generator = latent_synthsr_generator
#         self.latent_synthsr_prob = latent_synthsr_prob


#         # self.models = models
#         # self.dm_nb_inference_steps = dm_nb_inference_steps


#     def __len__(self):
#         return self._length

#     def load_segmentation(self, segmentation_npy_path):
#         segmentation = np.load(segmentation_npy_path)
#         # convert into one-hot encoding
#         # unique_labels = np.unique(segmentation)
#         # remove zero if it is in the unique labels (assuming zero is the background)
#         # if 0 in unique_labels:
#         #     unique_labels = unique_labels[unique_labels != 0]

#         unique_labels = list(
#             range(1, self.nb_seg_classes + 1)
#         )  # assuming classes are labeled from 1 to nb_classes

#         seg_onehot = []
#         for label in unique_labels:
#             seg_onehot.append(np.where(segmentation == label, 1.0, 0.0))
#         seg_onehot = np.stack(seg_onehot, axis=0)  # (C, D, H, W)

#         return torch.from_numpy(seg_onehot).float()

#     def load_pondered_segmentation(
#         self, segmentation_npy_path, current_modality, subject_id
#     ):
#         seg_list = []
#         for mod in self.used_modalities:
#             modified_path = segmentation_npy_path
#             if mod != current_modality:
#                 modified_path = segmentation_npy_path.replace(current_modality, mod)
#             # verify that the modified path exists
#             if os.path.exists(modified_path):
#                 seg_list.append(self.load_segmentation(modified_path))
#                 # print(f"Loaded segmentation from {modified_path} for modality {mod}")

#         # create weighted sum of the segmentations using random weights that sum to 1
#         if len(seg_list) > 1:
#             # if self.split == "train":
#             # print(f"{subject_id} nb segmentations {len(seg_list)} for modality {current_modality}")
#             if self.pondered_seg_generator is not None:
#                 gamma = torch.empty(len(seg_list)).exponential_(
#                     1.0, generator=self.pondered_seg_generator
#                 )
#             else:
#                 gamma = torch.ones(len(seg_list))
#             weights = gamma / gamma.sum()

#             seg_pondered = torch.zeros_like(seg_list[0])
#             for seg, w in zip(seg_list, weights):
#                 seg_pondered += w * seg
#         else:
#             seg_pondered = seg_list[0]
#         return seg_pondered

#     def __getitem__(self, index):
#         instance = self.train_data[index]

#         example = {}
#         example["subject_id"] = instance["subject_id"]

#         instance_latent = np.load(instance["latent_path"])
#         example["latent"] = torch.from_numpy(instance_latent)



#         example["modality"] = instance["modality"]
#         example["modality_idx"] = torch.tensor(instance["modality_idx"])
#         example["resolution"] = instance["resolution"]
#         example["resolution_idx"] = torch.tensor(instance["resolution_idx"])
#         example["org_img_path"] = instance["org_img_path"]

#         # segmentation_npy_path = instance["segmentation_npy_path"]
#         # segmentation = np.load(segmentation_npy_path)
#         # example["segmentation"] = torch.from_numpy(segmentation).long().unsqueeze(0)  # convert to long for one-hot encoding in the model

#         if self.use_pondered_segmentation:
#             example["segmentation"] = self.load_pondered_segmentation(
#                 instance["segmentation_npy_path"],
#                 instance["modality"],
#                 example["subject_id"],
#             )
#         else:
#             example["segmentation"] = self.load_segmentation(
#                 instance["segmentation_npy_path"]
#             )

#         # instance_latent_synthsr = np.load(instance["latent_synthsr_path"])
#         # example["latent_synthsr"] = torch.from_numpy(instance_latent_synthsr)
        
#         # load latent_synthsr depending th probability of each class and resolution
#         # max_prob_latent_synthsr = self.latent_synthsr_prob[instance["modality"]][instance["resolution"]]
#         load_latent_synthsr = True
#         # generate a random number between 0 and 1 using the latent_synthsr_generator
#         if self.latent_synthsr_generator is not None and self.latent_synthsr_prob is not None:
#             rand_num = torch.rand(1, generator=self.latent_synthsr_generator).item()
#             load_latent_synthsr = rand_num < self.latent_synthsr_prob
#         if load_latent_synthsr:
#             instance_latent_synthsr = np.load(instance["latent_synthsr_path"])
#             example["latent_synthsr"] = torch.from_numpy(instance_latent_synthsr)
#         else:
#             example["latent_synthsr"] = torch.zeros_like(example["latent"])



#         # ############
#         # self.models["noise_scheduler"].set_timesteps(
#         #     num_inference_steps=self.dm_nb_inference_steps,
#         #     input_img_size_numel=torch.prod(torch.tensor((48, 64, 48))),
#         # )
#         # if self.models is not None:
#         #     # regenerate the image
#         #     with torch.no_grad(), torch.amp.autocast("cuda"):
#         #         # segmentation_embedding = self.models["segmentation_encoder_model"](example["segmentation"].unsqueeze(0).to(device))
#         #         modality_embedding = self.models["modality_encoder_model"](
#         #             example["modality_idx"].unsqueeze(0).to(device)
#         #         )
#         #         resolution_embedding = self.models["resolution_encoder_model"](
#         #             example["resolution_idx"].unsqueeze(0).to(device)
#         #         )

#         #         all_timesteps = self.models["noise_scheduler"].timesteps
#         #         all_next_timesteps = torch.cat(
#         #             (all_timesteps[1:], torch.tensor([0], dtype=all_timesteps.dtype))
#         #         )

#         #         latents = example["latent"].unsqueeze(0).to(device)

#         #         for t, next_t in zip(all_timesteps, all_next_timesteps):
#         #             model_output = self.models["unet"](
#         #                 x=torch.cat(
#         #                     [latents, example["segmentation"].unsqueeze(0).to(device), example["latent_synthsr"].unsqueeze(0).to(device)],
#         #                     dim=1,
#         #                 ),
#         #                 timesteps=torch.Tensor((t,)).to(device),
#         #                 modallity_embedding=modality_embedding,
#         #                 resolution_embedding=resolution_embedding,
#         #             )

#         #             if not isinstance(self.models["noise_scheduler"], RFlowScheduler):
#         #                 latents, _ = self.models["noise_scheduler"].step(
#         #                     model_output, t, latents
#         #                 )
#         #             else:
#         #                 latents, _ = self.models["noise_scheduler"].step(
#         #                     model_output, t, latents, next_t
#         #                 )  # type: ignore

#         #         del model_output, modality_embedding, resolution_embedding
#         #         # torch.cuda.empty_cache()
#         #         example["latent_pred"] = latents.squeeze(0).detach().cpu()
#         #         example["latent_residuals"] = (example["latent"] - example["latent_pred"]).detach()
#         # ############

#         return example
